# ICS 591 Team 2 — Stage 2c: Fine-tuning the verification classifier (v1)

**Starting point:** the Stage 1 ViT-S/16 checkpoint, trained on RealWaste.

**Data:**
- RealWaste: about 4,700 labelled images, for material only.
- Our human-labelled local images from Stage 2b, for material, cleanliness and junk.

**Design: one shared ViT backbone with two heads (two output layers).**

| Head | Question | Classes | Learns from |
|---|---|---|---|
| **Material** | What is this? | `plastic`, `non_plastic`, `discard` (unusable photo) | RealWaste + local labels |
| **Cleanliness** | If plastic, is it clean? | `clean`, `contaminated` | local labels only |

Splitting the decision this way lets each question learn from the data that actually supports it. Material identification has thousands of examples. Cleanliness has only a few dozen local ones, so it is the weak part, and it is what pilot data will improve.

**Decision logic:**
- `discard` → **retake photo**
- `non_plastic` → **reject: wrong type**
- `plastic` + `contaminated` → **reject: contaminated**
- `plastic` + `clean` → **accept**
- Low confidence → **uncertain → human verifies**

**Evaluation:** the local test set (human-agreed labels from Stage 2b, never trained on) and the RealWaste test set (to check that material skill was not lost). The key result is **coverage versus accuracy**: how many pickups the model can decide alone, and how accurate those decisions are, with the rest going to a human.

**How to run:** Runtime → T4 GPU → Run all. It takes about 15–20 minutes.

In [ ]:
# 1. Setup
!pip -q install timm
import os, io, json, time, math, random, shutil, pathlib, copy
import numpy as np, pandas as pd
import torch, timm, torch.nn as nn
from PIL import Image, ImageFile
ImageFile.LOAD_TRUNCATED_IMAGES = True
from google.colab import drive
drive.mount('/content/drive')

LOCAL_OUT = pathlib.Path('/content/drive/MyDrive/ics591_local_data')
IMG_DRIVE = LOCAL_OUT / 'raw' / 'images'
S1_OUT    = pathlib.Path('/content/drive/MyDrive/ics591_model_selection/realwaste')
STAGE1_CKPT = S1_OUT / 'ViT-S_16.pt'
OUT = pathlib.Path('/content/drive/MyDrive/ics591_finetune'); OUT.mkdir(parents=True, exist_ok=True)
TIMM_ID = 'vit_small_patch16_224.augreg_in21k_ft_in1k'
for p in [IMG_DRIVE, STAGE1_CKPT]: assert p.exists(), f'missing: {p}'

DEV = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
assert DEV.type == 'cuda', 'Switch to a GPU runtime (Runtime > Change runtime type > T4 GPU)'
SEED = 42
random.seed(SEED); np.random.seed(SEED); torch.manual_seed(SEED)

MATERIAL = ['plastic', 'non_plastic', 'discard']
CLEAN = ['clean', 'contaminated']
DECISIONS = ['accept', 'reject_contaminated', 'reject_wrong_type', 'retake_photo']

EPOCHS = 8
BATCH = 32
LOCAL_SHARE = 0.35        # fraction of each batch drawn from local images (they are few but most relevant)
CLEAN_LOSS_WEIGHT = 1.0
TARGET_AUTO_ACC = 0.90    # operating point: model decides alone only where it is >= 90% accurate on validation

In [ ]:
# 2. Local labels from Stage 2b (human labels + model labels that passed the 95% bar)
TEAM = ['nana', 'kelvin', 'cajetan']
QUEUE, AUTO = LOCAL_OUT / 'al_queue.csv', LOCAL_OUT / 'al_auto_labels.csv'
def REV(p): return LOCAL_OUT / f'al_reviews_{p}.csv'
def to_model_label(l): return 'discard' if str(l).startswith('discard') else l
def load_reviews():
    parts = [pd.read_csv(REV(p)) for p in TEAM if REV(p).exists()]
    return pd.concat(parts, ignore_index=True) if parts else pd.DataFrame(columns=['file', 'label', 'pool', 'labeler', 'ts'])
def test_files():
    q = pd.read_csv(QUEUE); return set(q.loc[q['pool'] == 'test', 'file'])
def test_labels():
    r = load_reviews(); r = r[r['pool'] == 'test'].sort_values('ts').drop_duplicates(['file', 'labeler'], keep='last')
    out = []
    for f, g in r.groupby('file'):
        vc = g['label'].map(to_model_label).value_counts()
        if len(vc) == 1 or vc.iloc[0] > vc.iloc[1]: out.append({'file': f, 'label': vc.index[0]})
    return pd.DataFrame(out, columns=['file', 'label'])
def current_labels():
    r = load_reviews(); r = r[r['pool'] != 'test'].sort_values('ts').drop_duplicates('file', keep='last')
    human = pd.DataFrame({'file': r['file'], 'label': r['label'].map(to_model_label), 'source': 'human'})
    auto = pd.read_csv(AUTO) if AUTO.exists() else pd.DataFrame(columns=['file', 'label'])
    auto = pd.DataFrame({'file': auto['file'], 'label': auto['label'], 'source': 'model'})
    auto = auto[~auto['file'].isin(human['file'])]
    lab = pd.concat([human, auto], ignore_index=True)
    return lab[~lab['file'].isin(test_files())].reset_index(drop=True)

def heads_from_label(l):
    # 4-way local label -> (material index, cleanliness index or -1 = not applicable)
    return {'plastic_clean': (0, 0), 'plastic_contaminated': (0, 1),
            'non_plastic': (1, -1), 'discard': (2, -1)}[l]
def decision_from_label(l):
    return {'plastic_clean': 0, 'plastic_contaminated': 1, 'non_plastic': 2, 'discard': 3}[l]

from sklearn.model_selection import train_test_split
loc = current_labels()
loc_tr, loc_va = train_test_split(loc, test_size=0.2, stratify=loc['label'], random_state=SEED)
loc_te = test_labels()
print('Local train:'); print(loc_tr.groupby(['label', 'source']).size())
print('\nLocal validation:', loc_va['label'].value_counts().to_dict())
print('Local TEST (human-agreed, never trained on):', loc_te['label'].value_counts().to_dict())

LOCAL = pathlib.Path('/content/img_cache'); LOCAL.mkdir(exist_ok=True)
from concurrent.futures import ThreadPoolExecutor
need = [f for f in pd.concat([loc_tr, loc_va, loc_te])['file'] if not (LOCAL / f).exists()]
with ThreadPoolExecutor(16) as ex: list(ex.map(lambda f: shutil.copy(IMG_DRIVE / f, LOCAL / f), need))
print('local images cached')

In [ ]:
# 3. RealWaste: download and recreate the EXACT Stage 1 split (same seed), so Stage 1 test images stay unseen
DATASET = 'realwaste'
SPLIT = (0.70, 0.15, 0.15)
# 4. Download and locate the dataset
import zipfile, urllib.request

URLS = {
    'realwaste': 'https://archive.ics.uci.edu/static/public/908/realwaste.zip',
    'trashnet':  'https://huggingface.co/datasets/garythung/trashnet/resolve/main/dataset-resized.zip',
}
RAW = pathlib.Path('/content/raw'); RAW.mkdir(parents=True, exist_ok=True)
zip_path = RAW / f'{DATASET}.zip'
extract_dir = RAW / DATASET

if not zip_path.exists():
    print('Downloading', URLS[DATASET], '...')
    urllib.request.urlretrieve(URLS[DATASET], zip_path)
if not extract_dir.exists():
    with zipfile.ZipFile(zip_path) as z:
        z.extractall(extract_dir)
    for _ in range(2):  # some archives contain zips inside the zip
        for inner in list(extract_dir.rglob('*.zip')):
            with zipfile.ZipFile(inner) as z:
                z.extractall(inner.parent)
            inner.unlink()

IMG_EXT = {'.jpg', '.jpeg', '.png', '.bmp', '.webp'}
def has_images(d):
    return any(f.suffix.lower() in IMG_EXT for f in d.iterdir() if f.is_file())

# find the folder whose sub-folders are the classes
best, best_n = None, 0
for d in [extract_dir, *[p for p in extract_dir.rglob('*') if p.is_dir()]]:
    if '__MACOSX' in d.parts:
        continue
    n = sum(1 for c in d.iterdir() if c.is_dir() and has_images(c))
    if n > best_n:
        best, best_n = d, n
assert best_n >= 2, 'Could not find class folders - inspect /content/raw in the file browser'
SRC_ROOT = best
CLASSES = sorted(c.name for c in SRC_ROOT.iterdir() if c.is_dir() and has_images(c))
print('Class folder root:', SRC_ROOT)
print(len(CLASSES), 'classes:', CLASSES)

In [ ]:
# 3b. RealWaste: pre-resize images once (short side 256px) so training isn't bottlenecked by decoding
from PIL import Image
from concurrent.futures import ThreadPoolExecutor

DST_ROOT = pathlib.Path('/content/data') / DATASET
jobs = []
for ci, cname in enumerate(CLASSES):
    (DST_ROOT / cname).mkdir(parents=True, exist_ok=True)
    for f in sorted((SRC_ROOT / cname).iterdir()):
        if f.suffix.lower() in IMG_EXT:
            jobs.append((f, DST_ROOT / cname / (f.stem + '.jpg'), ci))

def resize_one(job):
    src, dst, ci = job
    if dst.exists():
        return (str(dst), ci)
    try:
        im = Image.open(src).convert('RGB')
        s = 256 / min(im.size)
        if s < 1:
            im = im.resize((round(im.width * s), round(im.height * s)), Image.BICUBIC)
        im.save(dst, quality=95)
        return (str(dst), ci)
    except Exception as e:
        print('Skipping unreadable image', src, e)
        return None

with ThreadPoolExecutor(8) as ex:
    SAMPLES = [r for r in ex.map(resize_one, jobs) if r is not None]
print(len(SAMPLES), 'images ready')

In [ ]:
# 3c. RealWaste: stratified train / validation / test split (same split for every model)
import numpy as np, pandas as pd
from sklearn.model_selection import train_test_split

paths = np.array([p for p, _ in SAMPLES]); labels = np.array([y for _, y in SAMPLES])
tr_p, tmp_p, tr_y, tmp_y = train_test_split(paths, labels, test_size=1 - SPLIT[0],
                                            stratify=labels, random_state=SEED)
rel_test = SPLIT[2] / (SPLIT[1] + SPLIT[2])
va_p, te_p, va_y, te_y = train_test_split(tmp_p, tmp_y, test_size=rel_test,
                                          stratify=tmp_y, random_state=SEED)
TRAIN, VAL, TEST = list(zip(tr_p, tr_y)), list(zip(va_p, va_y)), list(zip(te_p, te_y))

counts = pd.DataFrame({
    'train': np.bincount(tr_y, minlength=len(CLASSES)),
    'val':   np.bincount(va_y, minlength=len(CLASSES)),
    'test':  np.bincount(te_y, minlength=len(CLASSES)),
}, index=CLASSES)
counts.loc['TOTAL'] = counts.sum()


PLASTIC = next(i for i, c in enumerate(CLASSES) if 'plastic' in c.lower())

RW_CLASSES = CLASSES
prev = S1_OUT / 'split_counts.csv'
if prev.exists():
    same = pd.read_csv(prev, index_col=0).equals(counts.astype('int64'))
    print('RealWaste split identical to Stage 1:', same); assert same
print('RealWaste plastic class:', RW_CLASSES[PLASTIC])

In [ ]:
# 4. Build the combined training set, the model and the data pipeline
from torch.utils.data import Dataset, DataLoader, WeightedRandomSampler
from torchvision import transforms as T

def rw_items(split):   # RealWaste: material only (cleanliness unknown -> -1)
    return [(p, 0 if y == PLASTIC else 1, -1, 'realwaste') for p, y in split]
def local_items(df):
    return [(str(LOCAL / f), *heads_from_label(l), 'local') for f, l in zip(df['file'], df['label'])]

train_items = rw_items(TRAIN) + local_items(loc_tr)
is_local = np.array([it[3] == 'local' for it in train_items])
w = np.where(is_local, LOCAL_SHARE / is_local.sum(), (1 - LOCAL_SHARE) / (~is_local).sum())

ym = np.array([it[1] for it in train_items]); yc = np.array([it[2] for it in train_items])
def class_weights(y, w, n):
    p = np.array([w[y == c].sum() for c in range(n)]); p = p / p.sum()
    cw = np.where(p > 0, 1 / np.maximum(p, 1e-9), 0); return cw / cw[p > 0].mean()
WM = class_weights(ym, w, len(MATERIAL)); WC = class_weights(yc[yc >= 0], w[yc >= 0], len(CLEAN))
print('Training items:', len(train_items), f'({is_local.sum()} local, {(~is_local).sum()} RealWaste)')
print('Material class weights:', dict(zip(MATERIAL, WM.round(2))), '| cleanliness:', dict(zip(CLEAN, WC.round(2))))

class VerifierV1(nn.Module):
    def __init__(self, backbone):
        super().__init__()
        self.backbone = backbone
        d = backbone.num_features
        self.material = nn.Linear(d, len(MATERIAL))
        self.clean = nn.Linear(d, len(CLEAN))
    def forward(self, x):
        f = self.backbone(x)
        return self.material(f), self.clean(f)

def build_model():
    bb = timm.create_model(TIMM_ID, pretrained=False, num_classes=9)
    bb.load_state_dict(torch.load(STAGE1_CKPT, map_location='cpu'))
    bb.reset_classifier(0)              # backbone now outputs pooled features
    return VerifierV1(bb).to(DEV)

_cfg = timm.data.resolve_data_config({}, model=timm.create_model(TIMM_ID, pretrained=False))
NORM = T.Normalize(_cfg['mean'], _cfg['std'])
TRAIN_TF = T.Compose([T.RandomResizedCrop(224, scale=(0.5, 1.0)), T.RandomHorizontalFlip(),
                      T.ColorJitter(0.3, 0.3, 0.3), T.ToTensor(), NORM])
EVAL_TF = T.Compose([T.Resize(256), T.CenterCrop(224), T.ToTensor(), NORM])

class DS(Dataset):
    def __init__(self, items, tf): self.items, self.tf = items, tf
    def __len__(self): return len(self.items)
    def __getitem__(self, i):
        p, m, c, _ = self.items[i]
        return self.tf(Image.open(p).convert('RGB')), m, c

@torch.no_grad()
def predict(model, items):
    model.eval(); PM, PC = [], []
    for x, _, _ in DataLoader(DS(items, EVAL_TF), 64, num_workers=2):
        with torch.autocast('cuda', dtype=torch.float16):
            om, oc = model(x.to(DEV))
        PM.append(om.float().softmax(1).cpu()); PC.append(oc.float().softmax(1).cpu())
    return torch.cat(PM).numpy(), torch.cat(PC).numpy()

In [ ]:
# 5. Fine-tune
from sklearn.metrics import f1_score, accuracy_score, balanced_accuracy_score
model = build_model()
g = torch.Generator(); g.manual_seed(SEED)
sampler = WeightedRandomSampler(torch.tensor(w, dtype=torch.double), num_samples=len(train_items), replacement=True, generator=g)
train_dl = DataLoader(DS(train_items, TRAIN_TF), BATCH, sampler=sampler, num_workers=2, drop_last=True)
head_params = list(model.material.parameters()) + list(model.clean.parameters())
opt = torch.optim.AdamW([{'params': model.backbone.parameters(), 'lr': 3e-5},
                         {'params': head_params, 'lr': 1e-3}], weight_decay=0.05)
sched = torch.optim.lr_scheduler.OneCycleLR(opt, max_lr=[3e-5, 1e-3], total_steps=EPOCHS * len(train_dl), pct_start=0.1)
crit_m = nn.CrossEntropyLoss(weight=torch.tensor(WM, dtype=torch.float32, device=DEV), label_smoothing=0.1)
crit_c = nn.CrossEntropyLoss(weight=torch.tensor(WC, dtype=torch.float32, device=DEV), label_smoothing=0.1)
scaler = torch.amp.GradScaler('cuda')

val_local = local_items(loc_va); val_rw = rw_items(VAL)
history = []
for ep in range(1, EPOCHS + 1):
    model.train(); t0 = time.time(); tot = n = 0
    for x, m, c in train_dl:
        x, m, c = x.to(DEV), m.to(DEV), c.to(DEV)
        opt.zero_grad(set_to_none=True)
        with torch.autocast('cuda', dtype=torch.float16):
            om, oc = model(x)
            loss = crit_m(om, m)
            mask = c >= 0
            if mask.any(): loss = loss + CLEAN_LOSS_WEIGHT * crit_c(oc[mask], c[mask])
        scaler.scale(loss).backward(); scaler.step(opt); scaler.update(); sched.step()
        tot += loss.item() * x.size(0); n += x.size(0)
    pm, pc = predict(model, val_local); ym_v = np.array([i[1] for i in val_local]); yc_v = np.array([i[2] for i in val_local])
    pm_rw, _ = predict(model, val_rw); ym_rw = np.array([i[1] for i in val_rw])
    h = {'epoch': ep, 'loss': tot / n,
         'local_material_macro_f1': f1_score(ym_v, pm.argmax(1), average='macro', labels=range(3)),
         'local_clean_bal_acc': balanced_accuracy_score(yc_v[yc_v >= 0], pc[yc_v >= 0].argmax(1)) if (yc_v >= 0).any() else float('nan'),
         'realwaste_material_acc': accuracy_score(ym_rw, pm_rw.argmax(1))}
    history.append(h)
    print(f"ep {ep} | loss {h['loss']:.3f} | local material F1 {h['local_material_macro_f1']:.3f} | "
          f"local cleanliness bal-acc {h['local_clean_bal_acc']:.3f} | RealWaste material acc {h['realwaste_material_acc']:.3f} | {time.time() - t0:.0f}s")
torch.save(model.state_dict(), OUT / 'verifier_v1.pt')
pd.DataFrame(history).to_csv(OUT / 'training_history.csv', index=False)
print('saved', OUT / 'verifier_v1.pt')

In [ ]:
# 6. Calibrate (temperature per head, on validation) and choose the operating threshold (on validation)
from scipy.optimize import minimize_scalar
def fit_T(p, y):
    lp = np.log(np.clip(p, 1e-12, 1))
    def nll(t):
        z = lp / t; z -= z.max(1, keepdims=True); z -= np.log(np.exp(z).sum(1, keepdims=True))
        return -z[np.arange(len(y)), y].mean()
    return float(minimize_scalar(nll, bounds=(0.05, 10), method='bounded').x)
def apply_T(p, t):
    z = np.log(np.clip(p, 1e-12, 1)) / t; z -= z.max(1, keepdims=True); e = np.exp(z); return e / e.sum(1, keepdims=True)

def decide(pm, pc):
    # returns decision index (see DECISIONS) and decision confidence for each image
    mat = pm.argmax(1); cl = pc.argmax(1)
    dec = np.select([mat == 2, mat == 1, (mat == 0) & (cl == 1)], [3, 2, 1], default=0)
    conf = np.where(mat == 0, pm.max(1) * pc.max(1), pm.max(1))
    return dec, conf

def pick_threshold(conf, correct, target=TARGET_AUTO_ACC, min_n=15):
    for t in np.sort(np.unique(conf)):
        sel = conf >= t
        if sel.sum() < min_n: break
        if correct[sel].mean() >= target: return float(t)
    return 1.01   # never confident enough -> everything goes to a human

pm_v, pc_v = predict(model, val_local); pm_rwv, _ = predict(model, val_rw)
ym_v = np.array([i[1] for i in val_local]); yc_v = np.array([i[2] for i in val_local])
T_MAT = fit_T(np.vstack([pm_v, pm_rwv]), np.concatenate([ym_v, [i[1] for i in val_rw]]))
T_CLEAN = fit_T(pc_v[yc_v >= 0], yc_v[yc_v >= 0]) if (yc_v >= 0).sum() >= 5 else 1.0
dec_v, conf_v = decide(apply_T(pm_v, T_MAT), apply_T(pc_v, T_CLEAN))
yd_v = loc_va['label'].map(decision_from_label).values
THRESH = pick_threshold(conf_v, dec_v == yd_v)
print(f'Temperatures: material {T_MAT:.3f}, cleanliness {T_CLEAN:.3f}')
print(f'Operating threshold (validation, >= {TARGET_AUTO_ACC:.0%} accuracy on auto-decided cases): {THRESH:.3f}')

In [ ]:
# 7. Evaluate on the local TEST set (human-agreed, never trained on) and the RealWaste test set
from sklearn.metrics import confusion_matrix
test_local = local_items(loc_te)
pm_t, pc_t = predict(model, test_local); pm_t, pc_t = apply_T(pm_t, T_MAT), apply_T(pc_t, T_CLEAN)
dec_t, conf_t = decide(pm_t, pc_t); yd_t = loc_te['label'].map(decision_from_label).values
ym_t = np.array([i[1] for i in test_local]); yc_t = np.array([i[2] for i in test_local])
correct = dec_t == yd_t

rng = np.random.default_rng(SEED)
boot = [correct[rng.integers(0, len(correct), len(correct))].mean() for _ in range(2000)]
auto = conf_t >= THRESH
R = {
  'local_test_n': int(len(yd_t)),
  'decision_accuracy_all': float(correct.mean()),
  'decision_accuracy_all_95ci': [float(np.percentile(boot, 2.5)), float(np.percentile(boot, 97.5))],
  'decision_macro_f1_all': float(f1_score(yd_t, dec_t, average='macro', labels=range(4))),
  'material_accuracy': float(accuracy_score(ym_t, pm_t.argmax(1))),
  'material_macro_f1': float(f1_score(ym_t, pm_t.argmax(1), average='macro', labels=range(3))),
  'cleanliness_accuracy_on_plastic': float((pc_t[yc_t >= 0].argmax(1) == yc_t[yc_t >= 0]).mean()),
  'cleanliness_test_n': int((yc_t >= 0).sum()),
  'threshold': THRESH, 'auto_decided_share': float(auto.mean()),
  'auto_decided_accuracy': float(correct[auto].mean()) if auto.any() else None,
  'sent_to_human_share': float(1 - auto.mean()),
}
# no-forgetting check on RealWaste test (material: plastic vs non-plastic, ignoring 'discard')
test_rw = rw_items(TEST); pm_rwt, _ = predict(model, test_rw); y_rwt = np.array([i[1] for i in test_rw])
R['realwaste_test_material_acc'] = float((pm_rwt[:, :2].argmax(1) == y_rwt).mean())

# baseline: Stage 1 model with no local fine-tuning, plastic vs non-plastic on local test (non-discard)
s1 = timm.create_model(TIMM_ID, pretrained=False, num_classes=9); s1.load_state_dict(torch.load(STAGE1_CKPT, map_location='cpu')); s1 = s1.to(DEV).eval()
nd = [it for it in test_local if it[1] != 2]; ynd = np.array([it[1] for it in nd])
with torch.no_grad():
    P = []
    for x, _, _ in DataLoader(DS(nd, EVAL_TF), 64, num_workers=2):
        with torch.autocast('cuda', dtype=torch.float16): P.append(s1(x.to(DEV)).float().softmax(1).cpu())
    s1_pred = np.where(torch.cat(P).numpy().argmax(1) == PLASTIC, 0, 1)
nd_mask = ym_t != 2
R['baseline_stage1_plastic_vs_non_acc_local'] = float((s1_pred == ynd).mean())
R['v1_plastic_vs_non_acc_local'] = float((pm_t[nd_mask][:, :2].argmax(1) == ym_t[nd_mask]).mean())

(OUT / 'v1_test_results.json').write_text(json.dumps(R, indent=2))
print(json.dumps(R, indent=2))
cm = pd.DataFrame(confusion_matrix(yd_t, dec_t, labels=range(4)), index=['true ' + d for d in DECISIONS],
                  columns=['pred ' + d for d in DECISIONS])
cm.to_csv(OUT / 'v1_decision_confusion.csv'); cm

In [ ]:
# 8. Coverage vs accuracy — the key chart for the human-in-the-loop argument
import matplotlib.pyplot as plt
order = np.argsort(-conf_t); cum = np.cumsum(correct[order]) / np.arange(1, len(order) + 1)
cov = np.arange(1, len(order) + 1) / len(order)
rows = []
for c in [0.25, 0.5, 0.75, 1.0]:
    k = max(1, int(round(c * len(order)))); rows.append({'model decides alone': f'{c:.0%}', 'accuracy on those': f'{cum[k - 1]:.1%}',
                                                         'sent to human': f'{1 - c:.0%}'})
rc = pd.DataFrame(rows); rc.to_csv(OUT / 'v1_coverage_table.csv', index=False)

fig, ax = plt.subplots(1, 2, figsize=(14, 5))
ax[0].plot(cov, cum, lw=2); ax[0].axhline(TARGET_AUTO_ACC, ls='--', c='gray', label=f'{TARGET_AUTO_ACC:.0%} target')
if auto.any(): ax[0].scatter([auto.mean()], [correct[auto].mean()], c='red', zorder=3, label=f'operating point (threshold {THRESH:.2f})')
ax[0].set_xlabel('Share of pickups the model decides alone (coverage)'); ax[0].set_ylabel('Accuracy on those decisions')
ax[0].set_title('Local test set: coverage vs accuracy'); ax[0].set_ylim(0, 1.02); ax[0].legend()
h = pd.DataFrame(history)
ax[1].plot(h['epoch'], h['local_material_macro_f1'], marker='o', label='local material macro-F1 (val)')
ax[1].plot(h['epoch'], h['local_clean_bal_acc'], marker='o', label='local cleanliness balanced acc (val)')
ax[1].plot(h['epoch'], h['realwaste_material_acc'], marker='o', label='RealWaste material acc (val)')
ax[1].set_xlabel('epoch'); ax[1].set_ylim(0, 1.02); ax[1].set_title('Training progress'); ax[1].legend()
plt.tight_layout(); plt.savefig(OUT / 'v1_coverage_and_training.png', dpi=150); plt.show()
rc

In [ ]:
# 9. Export config + a ready-to-use inference function for the backend team
config = {'model': 'VerifierV1 (ViT-S/16 backbone + material head + cleanliness head)', 'timm_id': TIMM_ID,
          'weights_file': 'verifier_v1.pt', 'material_classes': MATERIAL, 'clean_classes': CLEAN,
          'decisions': DECISIONS + ['uncertain_human_review'], 'temperature_material': T_MAT,
          'temperature_clean': T_CLEAN, 'auto_decision_threshold': THRESH, 'image_size': 224,
          'normalize_mean': list(_cfg['mean']), 'normalize_std': list(_cfg['std'])}
(OUT / 'verifier_v1_config.json').write_text(json.dumps(config, indent=2))

def verify(image_path, model=model, cfg=config):
    # Classify one pickup photo. Returns the decision the backend should act on.
    x = EVAL_TF(Image.open(image_path).convert('RGB')).unsqueeze(0).to(DEV)
    with torch.no_grad():
        om, oc = model.eval()(x)
    pm = apply_T(om.float().softmax(1).cpu().numpy(), cfg['temperature_material'])
    pc = apply_T(oc.float().softmax(1).cpu().numpy(), cfg['temperature_clean'])
    d, c = decide(pm, pc)
    decision = DECISIONS[d[0]] if c[0] >= cfg['auto_decision_threshold'] else 'uncertain_human_review'
    return {'decision': decision, 'suggested': DECISIONS[d[0]], 'confidence': round(float(c[0]), 3),
            'material': dict(zip(MATERIAL, pm[0].round(3).tolist())), 'cleanliness': dict(zip(CLEAN, pc[0].round(3).tolist()))}

# demo on three local test images
for f in loc_te['file'].sample(3, random_state=1):
    print(f, '| true:', loc_te.set_index('file').loc[f, 'label'], '|', verify(LOCAL / f))
print('\nSaved to', OUT, ':', sorted(p.name for p in OUT.iterdir()))